In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7377657bbe00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x737765600980>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

/tmp/ipykernel_86158/4150212194.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.Amnswer all the question to the nest of your ability",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [ ]:
chain.invoke({"messages": [HumanMessage(content="Hi My name is Anas")]})

AIMessage(content='Hello Anas! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hi My name is Anas". They haven\'t asked a question yet. We should respond politely, perhaps ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 98, 'total_tokens': 150, 'completion_time': 0.053910086, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.004753856, 'prompt_tokens_details': None, 'queue_time': 0.372663584, 'total_time': 0.058663942}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11746-a3aa-7671-8ced-9ff0380075f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 52, 'total_tokens': 150, 'output_token_details': {'reasoning': 30}})

In [ ]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [ ]:
config = {"configurable": {"session_id": "chat1"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is Anas")], config=config
)
response

/tmp/ipykernel_86158/4150212194.py:10: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=ChatMessageHistory()


AIMessage(content='Hello Anas! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says: "Hi My name is Anas". Likely greeting. We should respond politely. We might ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 98, 'total_tokens': 148, 'completion_time': 0.083031154, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.050077751, 'prompt_tokens_details': None, 'queue_time': 0.405498464, 'total_time': 0.133108905}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11747-f492-7341-9e0e-9d0b520c2bcd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 50, 'total_tokens': 148, 'output_token_details': {'reasoning': 28}})

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
response.content

'Your name is Anas.'

In [10]:
## Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [15]:
response=chain.invoke({"messages":[HumanMessage(content="Hi my name is Anas")],"language":"Italian"})
response.content

'Ciao Anas! Come posso aiutarti oggi?'

In [16]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/home/anas/LEARNING/GEN-AI/LangChain/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config = {"configurable": {"session_id": "chat2"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Anas")],"language":"Italian"},
    config=config
)
repsonse.content

'Ciao Anas! Come posso aiutarti oggi?'

In [18]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Italian"},
    config=config,
)

In [19]:
response.content

'Il tuo nome è Anas.'